In [ ]:
import os
import sys
import time
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, roc_curve, confusion_matrix)

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['figure.titlesize'] = 16

def main():
    print("=" * 70)
    print("  EXPERIMENT 4: BINARY CLASSIFICATION USING LINEAR & KERNEL MODELS")
    print("=" * 70)

    # 1. Directory Setup
    base_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    dataset_path = os.path.join(base_dir, 'dataset', 'spambase_csv.csv')
    plots_dir = os.path.join(base_dir, 'output_plots')
    os.makedirs(plots_dir, exist_ok=True)

    # 2. Load Dataset
    print(f"\n[Step 1] Loading dataset from: {dataset_path}")
    df = pd.read_csv(dataset_path)
    print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")

    target_col = 'class' if 'class' in df.columns else df.columns[-1]
    X = df.drop(columns=[target_col])
    y = df[target_col]

    missing_count = df.isnull().sum().sum()
    print(f"Missing values count: {missing_count}")

    class_counts = y.value_counts().to_dict()
    spam_cnt = class_counts.get(1, 0)
    ham_cnt = class_counts.get(0, 0)
    print(f"Class breakdown: Ham (0) = {ham_cnt} ({ham_cnt/len(y)*100:.2f}%), Spam (1) = {spam_cnt} ({spam_cnt/len(y)*100:.2f}%)")

    # 3. Exploratory Data Analysis & Plots
    print("\n[Step 2] Performing EDA and generating EDA plots...")
    
    # Plot 1: Class Distribution
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    colors = ['#2b5c8f', '#d9534f']
    
    sns.barplot(x=['Ham (0)', 'Spam (1)'], y=[ham_cnt, spam_cnt], palette=colors, ax=axes[0], hue=['Ham (0)', 'Spam (1)'], legend=False)
    axes[0].set_title('Class Counts (Ham vs Spam)', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Number of Emails')
    for p in axes[0].patches:
        axes[0].annotate(f"{int(p.get_height())}\n({p.get_height()/len(y)*100:.1f}%)",
                         (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                         ha='center', va='center', color='white', fontweight='bold', fontsize=12)
        
    axes[1].pie([ham_cnt, spam_cnt], labels=['Ham (0)', 'Spam (1)'], autopct='%1.1f%%',
               startangle=90, colors=colors, explode=(0.05, 0), textprops={'fontsize': 12, 'weight': 'bold'})
    axes[1].set_title('Class Proportion', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, '01_class_distribution.png'), dpi=300)
    plt.close()

    # Plot 2: Top Correlated Features
    correlations = df.corr()[target_col].drop(target_col).abs().sort_values(ascending=False)
    top_15_corr = correlations.head(15)

    plt.figure(figsize=(10, 6))
    sns.barplot(x=top_15_corr.values, y=top_15_corr.index, palette='viridis', hue=top_15_corr.index, legend=False)
    plt.title('Top 15 Features by Absolute Correlation with Target (Spam/Ham)', fontsize=14, fontweight='bold')
    plt.xlabel('Absolute Correlation Coefficient')
    plt.ylabel('Feature Name / Index')
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, '02_feature_correlation.png'), dpi=300)
    plt.close()

    # 4. Train-Test Split & Standardization
    print("\n[Step 3] Splitting data (80% Train, 20% Test) and Standardizing features...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f"Training set: {X_train_scaled.shape[0]} samples | Test set: {X_test_scaled.shape[0]} samples")

    # 5. Baseline Logistic Regression
    print("\n[Step 4] Training Baseline Logistic Regression...")
    t0 = time.time()
    lr_base = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
    lr_base.fit(X_train_scaled, y_train)
    t_lr_base = time.time() - t0

    y_pred_lr_base = lr_base.predict(X_test_scaled)
    y_prob_lr_base = lr_base.predict_proba(X_test_scaled)[:, 1]

    acc_lr_base = accuracy_score(y_test, y_pred_lr_base)
    prec_lr_base = precision_score(y_test, y_pred_lr_base)
    rec_lr_base = recall_score(y_test, y_pred_lr_base)
    f1_lr_base = f1_score(y_test, y_pred_lr_base)
    auc_lr_base = roc_auc_score(y_test, y_prob_lr_base)

    print(f"Baseline LR -> Acc: {acc_lr_base:.4f}, Prec: {prec_lr_base:.4f}, Rec: {rec_lr_base:.4f}, F1: {f1_lr_base:.4f}, AUC: {auc_lr_base:.4f}, Time: {t_lr_base:.4f}s")

    # 6. Hyperparameter Tuning - Logistic Regression
    print("\n[Step 5] Tuning Logistic Regression (Grid Search & Randomized Search)...")
    
    cv_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    lr_param_grid = [
        {'penalty': ['l1', 'l2'], 'C': [0.01, 0.1, 1, 10, 100], 'solver': ['liblinear']}
    ]

    # Grid Search LR
    t0 = time.time()
    lr_gs = GridSearchCV(LogisticRegression(random_state=RANDOM_STATE, max_iter=1000), 
                         lr_param_grid, cv=cv_stratified, scoring='accuracy', n_jobs=1)
    lr_gs.fit(X_train_scaled, y_train)
    t_lr_gs = time.time() - t0
    best_lr_gs = lr_gs.best_estimator_
    best_lr_gs_acc = lr_gs.best_score_
    print(f"LR Grid Search Best Params: {lr_gs.best_params_}")
    print(f"LR Grid Search Best CV Accuracy: {best_lr_gs_acc:.4f} (Time: {t_lr_gs:.2f}s)")

    # Randomized Search LR
    t0 = time.time()
    lr_rs = RandomizedSearchCV(LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
                               lr_param_grid, n_iter=6, cv=cv_stratified, scoring='accuracy', 
                               random_state=RANDOM_STATE, n_jobs=1)
    lr_rs.fit(X_train_scaled, y_train)
    t_lr_rs = time.time() - t0
    best_lr_rs_acc = lr_rs.best_score_
    print(f"LR Randomized Search Best Params: {lr_rs.best_params_}")
    print(f"LR Randomized Search Best CV Accuracy: {best_lr_rs_acc:.4f} (Time: {t_lr_rs:.2f}s)")

    # Best Tuned LR Test Evaluation
    y_pred_lr_best = best_lr_gs.predict(X_test_scaled)
    y_prob_lr_best = best_lr_gs.predict_proba(X_test_scaled)[:, 1]
    acc_lr_best = accuracy_score(y_test, y_pred_lr_best)
    prec_lr_best = precision_score(y_test, y_pred_lr_best)
    rec_lr_best = recall_score(y_test, y_pred_lr_best)
    f1_lr_best = f1_score(y_test, y_pred_lr_best)
    auc_lr_best = roc_auc_score(y_test, y_prob_lr_best)

    # 7. SVM Kernel Comparison (Baseline Kernels)
    print("\n[Step 6] Training SVM with Different Kernels (Linear, Poly, RBF, Sigmoid)...")
    kernels = ['linear', 'poly', 'rbf', 'sigmoid']
    svm_kernel_results = []
    svm_kernel_models = {}

    for k in kernels:
        t0 = time.time()
        svm_model = SVC(kernel=k, C=1.0, gamma='scale', probability=True, random_state=RANDOM_STATE)
        svm_model.fit(X_train_scaled, y_train)
        t_fit = time.time() - t0

        y_pred = svm_model.predict(X_test_scaled)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        prob = svm_model.predict_proba(X_test_scaled)[:, 1]
        auc = roc_auc_score(y_test, prob)

        svm_kernel_results.append({
            'Kernel': k.capitalize(),
            'Accuracy': float(acc),
            'Precision': float(prec),
            'Recall': float(rec),
            'F1 Score': float(f1),
            'ROC AUC': float(auc),
            'Training Time (s)': float(t_fit)
        })
        svm_kernel_models[k] = svm_model
        print(f"SVM ({k.capitalize()}) -> Acc: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}, Time: {t_fit:.4f}s")

    svm_kernel_df = pd.DataFrame(svm_kernel_results)

    # 8. SVM Hyperparameter Tuning (Grid Search & Randomized Search)
    print("\n[Step 7] Tuning SVM Hyperparameters (Grid Search & Randomized Search)...")
    
    svm_param_grid = [
        {'kernel': ['linear'], 'C': [0.1, 1, 10]},
        {'kernel': ['rbf', 'sigmoid'], 'C': [0.1, 1, 10], 'gamma': ['scale', 'auto']},
        {'kernel': ['poly'], 'C': [0.1, 1, 10], 'gamma': ['scale'], 'degree': [2, 3]}
    ]

    # Grid Search SVM
    t0 = time.time()
    svm_gs = GridSearchCV(SVC(random_state=RANDOM_STATE), 
                          svm_param_grid, cv=cv_stratified, scoring='accuracy', n_jobs=1)
    svm_gs.fit(X_train_scaled, y_train)
    t_svm_gs = time.time() - t0
    best_svm_params = svm_gs.best_params_
    best_svm_gs_acc = svm_gs.best_score_
    print(f"SVM Grid Search Best Params: {best_svm_params}")
    print(f"SVM Grid Search Best CV Accuracy: {best_svm_gs_acc:.4f} (Time: {t_svm_gs:.2f}s)")

    # Randomized Search SVM
    t0 = time.time()
    svm_rs = RandomizedSearchCV(SVC(random_state=RANDOM_STATE),
                                svm_param_grid, n_iter=5, cv=cv_stratified, scoring='accuracy',
                                random_state=RANDOM_STATE, n_jobs=1)
    svm_rs.fit(X_train_scaled, y_train)
    t_svm_rs = time.time() - t0
    best_svm_rs_acc = svm_rs.best_score_
    print(f"SVM Randomized Search Best Params: {svm_rs.best_params_}")
    print(f"SVM Randomized Search Best CV Accuracy: {best_svm_rs_acc:.4f} (Time: {t_svm_rs:.2f}s)")

    # Fit final best SVM with probability=True for ROC Curve evaluation
    best_svm_model = SVC(**best_svm_params, probability=True, random_state=RANDOM_STATE)
    best_svm_model.fit(X_train_scaled, y_train)

    # Best Tuned SVM Test Evaluation
    y_pred_svm_best = best_svm_model.predict(X_test_scaled)
    y_prob_svm_best = best_svm_model.predict_proba(X_test_scaled)[:, 1]
    acc_svm_best = accuracy_score(y_test, y_pred_svm_best)
    prec_svm_best = precision_score(y_test, y_pred_svm_best)
    rec_svm_best = recall_score(y_test, y_pred_svm_best)
    f1_svm_best = f1_score(y_test, y_pred_svm_best)
    auc_svm_best = roc_auc_score(y_test, y_prob_svm_best)

    # 9. 5-Fold Cross-Validation Analysis
    print("\n[Step 8] Performing 5-Fold Cross-Validation for Best LR and Best SVM...")
    
    lr_cv_scores = cross_val_score(best_lr_gs, X_train_scaled, y_train, cv=cv_stratified, scoring='accuracy')
    svm_cv_scores = cross_val_score(best_svm_model, X_train_scaled, y_train, cv=cv_stratified, scoring='accuracy')

    folds = [f"Fold {i+1}" for i in range(5)]
    kfold_df = pd.DataFrame({
        'Fold': folds,
        'Logistic Regression': lr_cv_scores,
        'SVM': svm_cv_scores
    })
    kfold_avg = pd.DataFrame({
        'Fold': ['Average'],
        'Logistic Regression': [np.mean(lr_cv_scores)],
        'SVM': [np.mean(svm_cv_scores)]
    })
    kfold_full_df = pd.concat([kfold_df, kfold_avg], ignore_index=True)

    print("\n5-Fold CV Accuracy Breakdown:")
    print(kfold_full_df.to_string(index=False))

    # 10. Save Detailed Visualization Plots
    print("\n[Step 9] Generating comprehensive evaluation plots...")

    # Plot 3: Confusion Matrices
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    cms = [
        (confusion_matrix(y_test, y_pred_lr_base), "Baseline Logistic Regression", axes[0, 0]),
        (confusion_matrix(y_test, y_pred_lr_best), "Tuned Logistic Regression", axes[0, 1]),
        (confusion_matrix(y_test, svm_kernel_models['rbf'].predict(X_test_scaled)), "Baseline SVM (RBF Kernel)", axes[1, 0]),
        (confusion_matrix(y_test, y_pred_svm_best), "Tuned SVM (Best Model)", axes[1, 1])
    ]
    for cm, title, ax in cms:
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                    xticklabels=['Ham (0)', 'Spam (1)'], yticklabels=['Ham (0)', 'Spam (1)'],
                    annot_kws={'size': 14, 'weight': 'bold'})
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('Predicted Label')
        ax.set_ylabel('True Label')
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, '03_confusion_matrices.png'), dpi=300)
    plt.close()

    # Plot 4: ROC Curves
    plt.figure(figsize=(10, 8))
    
    # LR ROC
    fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr_best)
    plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {auc_lr_best:.4f})', linewidth=2.5, color='#1f77b4')

    # SVM Kernels ROC
    colors_svm = {'linear': '#ff7f0e', 'poly': '#2ca02c', 'rbf': '#d62728', 'sigmoid': '#9467bd'}
    for k in kernels:
        prob = svm_kernel_models[k].predict_proba(X_test_scaled)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, prob)
        auc_val = roc_auc_score(y_test, prob)
        plt.plot(fpr, tpr, label=f'SVM ({k.capitalize()}) (AUC = {auc_val:.4f})', linewidth=1.8, linestyle='--', color=colors_svm[k])

    plt.plot([0, 1], [0, 1], 'k--', label='Random Chance (AUC = 0.5000)', linewidth=1.5)
    plt.title('Receiver Operating Characteristic (ROC) Curves', fontsize=15, fontweight='bold')
    plt.xlabel('False Positive Rate (1 - Specificity)')
    plt.ylabel('True Positive Rate (Sensitivity / Recall)')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, '04_roc_curves.png'), dpi=300)
    plt.close()

    # Plot 5: SVM Kernel-wise Performance
    fig, ax1 = plt.subplots(figsize=(10, 6))
    x = np.arange(len(kernels))
    width = 0.35

    rects1 = ax1.bar(x - width/2, svm_kernel_df['Accuracy'], width, label='Accuracy', color='#2b5c8f')
    rects2 = ax1.bar(x + width/2, svm_kernel_df['F1 Score'], width, label='F1 Score', color='#d9534f')

    ax1.set_ylabel('Score', color='black', fontsize=12)
    ax1.set_title('SVM Performance & Training Time by Kernel Type', fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels([k.capitalize() for k in kernels], fontsize=11, fontweight='bold')
    ax1.set_ylim(0.7, 1.0)
    ax1.legend(loc='upper left')

    # Secondary axis for training time
    ax2 = ax1.twinx()
    ax2.plot(x, svm_kernel_df['Training Time (s)'], color='#2ca02c', marker='o', linewidth=2.5, label='Training Time (s)')
    ax2.set_ylabel('Training Time (seconds)', color='#2ca02c', fontsize=12)
    ax2.grid(False)

    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, '05_svm_kernel_comparison.png'), dpi=300)
    plt.close()

    # Plot 6: LR Hyperparameter Tuning Curve (C vs Accuracy for L1 & L2)
    c_vals = [0.01, 0.1, 1, 10, 100]
    l1_scores = []
    l2_scores = []

    for c in c_vals:
        lr_l1 = LogisticRegression(penalty='l1', C=c, solver='liblinear', random_state=RANDOM_STATE, max_iter=1000)
        lr_l2 = LogisticRegression(penalty='l2', C=c, solver='liblinear', random_state=RANDOM_STATE, max_iter=1000)
        
        score_l1 = np.mean(cross_val_score(lr_l1, X_train_scaled, y_train, cv=cv_stratified))
        score_l2 = np.mean(cross_val_score(lr_l2, X_train_scaled, y_train, cv=cv_stratified))
        
        l1_scores.append(score_l1)
        l2_scores.append(score_l2)

    plt.figure(figsize=(9, 6))
    plt.plot(c_vals, l1_scores, marker='o', linewidth=2.5, label='L1 Regularization (Lasso)', color='#d9534f')
    plt.plot(c_vals, l2_scores, marker='s', linewidth=2.5, label='L2 Regularization (Ridge)', color='#2b5c8f')
    plt.xscale('log')
    plt.title('Logistic Regression: Effect of Regularization Parameter C', fontsize=14, fontweight='bold')
    plt.xlabel('Inverse Regularization Strength (C) [Log Scale]')
    plt.ylabel('5-Fold Cross-Validation Accuracy')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, '06_hyperparameter_tuning_lr.png'), dpi=300)
    plt.close()

    # Plot 7: SVM Tuning Heatmap (C vs Kernel Accuracy)
    c_svm_list = [0.1, 1, 10]
    kernel_list = ['linear', 'rbf', 'poly', 'sigmoid']
    svm_grid_matrix = np.zeros((len(kernel_list), len(c_svm_list)))

    for i, k in enumerate(kernel_list):
        for j, c in enumerate(c_svm_list):
            clf = SVC(kernel=k, C=c, gamma='scale', random_state=RANDOM_STATE)
            score = np.mean(cross_val_score(clf, X_train_scaled, y_train, cv=cv_stratified))
            svm_grid_matrix[i, j] = score

    plt.figure(figsize=(9, 6))
    sns.heatmap(svm_grid_matrix, annot=True, fmt='.4f', cmap='YlGnBu',
                xticklabels=c_svm_list, yticklabels=[k.capitalize() for k in kernel_list])
    plt.title('SVM 5-Fold CV Accuracy Grid (Kernel vs Regularization C)', fontsize=14, fontweight='bold')
    plt.xlabel('Regularization Parameter (C)')
    plt.ylabel('Kernel Type')
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, '07_hyperparameter_tuning_svm.png'), dpi=300)
    plt.close()

    # Plot 8: K-Fold CV Comparison Plot
    plt.figure(figsize=(9, 5))
    plt.plot(folds, lr_cv_scores, marker='o', linewidth=2.5, label='Logistic Regression', color='#1f77b4')
    plt.plot(folds, svm_cv_scores, marker='s', linewidth=2.5, label='SVM (Best RBF)', color='#ff7f0e')
    plt.axhline(y=np.mean(lr_cv_scores), color='#1f77b4', linestyle='--', alpha=0.7, label=f'LR Mean ({np.mean(lr_cv_scores):.4f})')
    plt.axhline(y=np.mean(svm_cv_scores), color='#ff7f0e', linestyle='--', alpha=0.7, label=f'SVM Mean ({np.mean(svm_cv_scores):.4f})')
    plt.title('5-Fold Cross-Validation Accuracy across Folds', fontsize=14, fontweight='bold')
    plt.xlabel('CV Fold Index')
    plt.ylabel('Validation Accuracy')
    plt.ylim(0.88, 0.96)
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, '08_kfold_cv_comparison.png'), dpi=300)
    plt.close()

    # 11. Consolidate and Save Metrics to JSON
    summary_data = {
        'tuning_results': [
            {
                'Model': 'Logistic Regression',
                'Search Method': 'Grid Search',
                'Best Parameters': str(lr_gs.best_params_),
                'Best CV Accuracy': float(best_lr_gs_acc)
            },
            {
                'Model': 'Logistic Regression',
                'Search Method': 'Randomized Search',
                'Best Parameters': str(lr_rs.best_params_),
                'Best CV Accuracy': float(best_lr_rs_acc)
            },
            {
                'Model': 'Support Vector Machine',
                'Search Method': 'Grid Search',
                'Best Parameters': str(best_svm_params),
                'Best CV Accuracy': float(best_svm_gs_acc)
            },
            {
                'Model': 'Support Vector Machine',
                'Search Method': 'Randomized Search',
                'Best Parameters': str(svm_rs.best_params_),
                'Best CV Accuracy': float(best_svm_rs_acc)
            }
        ],
        'lr_performance': {
            'Accuracy': float(acc_lr_best),
            'Precision': float(prec_lr_best),
            'Recall': float(rec_lr_best),
            'F1 Score': float(f1_lr_best),
            'ROC AUC': float(auc_lr_best),
            'Training Time (s)': float(t_lr_gs)
        },
        'svm_kernel_wise': svm_kernel_df.to_dict(orient='records'),
        'svm_best_performance': {
            'Accuracy': float(acc_svm_best),
            'Precision': float(prec_svm_best),
            'Recall': float(rec_svm_best),
            'F1 Score': float(f1_svm_best),
            'ROC AUC': float(auc_svm_best),
            'Training Time (s)': float(t_svm_gs)
        },
        'kfold_results': {
            'folds': folds,
            'lr_scores': [float(x) for x in lr_cv_scores],
            'svm_scores': [float(x) for x in svm_cv_scores],
            'lr_avg': float(np.mean(lr_cv_scores)),
            'svm_avg': float(np.mean(svm_cv_scores))
        }
    }

    metrics_json_path = os.path.join(base_dir, 'code', 'experiment4_results.json')
    with open(metrics_json_path, 'w') as f:
        json.dump(summary_data, f, indent=4)

    print(f"\n[Done] Pipeline complete! Results saved to {metrics_json_path}")
    print("=" * 70)

if __name__ == '__main__':
    main()
